In [2]:
import sys
import torch
from pathlib import Path
from hydra import initialize_config_dir, compose


PROJECT_ROOT = Path.cwd().parent
if PROJECT_ROOT not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from tracers.datasets import (
    AutoOsu4KDataset,
    split_song_ids,
    make_dataloader,
)

ROOT = PROJECT_ROOT / "dataset"

AUDIO_DIR = ROOT / "audio"
BEATMAP_DIR = ROOT / "beatmap" / "4keys"
CONFIG_DIR = PROJECT_ROOT / "conf"


In [3]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(device)

with initialize_config_dir(
    version_base=None,
    config_dir=str(CONFIG_DIR),
):
    cfg = compose(config_name="train")


experiment_config = cfg.experiment
hyperparams = cfg.hyperparams

segment_frames = hyperparams["segment_frames"]
batch_size = hyperparams["batch_size"]



cuda


In [4]:

base_dataset = AutoOsu4KDataset(
    beatmap_dir=BEATMAP_DIR,
    audio_dir=AUDIO_DIR,
    segment_frames=segment_frames,
    crop_mode="random",
    audio_cache_size=8,
)

train_ids, val_ids, test_ids = split_song_ids(
    base_dataset.song_ids,
    val_fraction=0.10,
    test_fraction=0.10,
    seed=42,
)



In [5]:


train_dataset = AutoOsu4KDataset(
    BEATMAP_DIR,
    AUDIO_DIR,
    segment_frames=segment_frames,
    crop_mode="random",
    crops_per_chart=2,
    song_ids=train_ids,
)

val_dataset = AutoOsu4KDataset(
    BEATMAP_DIR,
    AUDIO_DIR,
    segment_frames=segment_frames,
    crop_mode="all",
    song_ids=val_ids,
)

test_dataset = AutoOsu4KDataset(
    BEATMAP_DIR,
    AUDIO_DIR,
    segment_frames=segment_frames,
    crop_mode="all",
    song_ids=test_ids,
)



In [6]:
train_loader = make_dataloader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

val_loader = make_dataloader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

test_loader = make_dataloader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

In [7]:
from tracers.models.transformer import TracerNpModel

model = TracerNpModel(hyperparams).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=hyperparams["learning_rate"],
    weight_decay=hyperparams["weight_decay"]
)

In [8]:
from tracers.training.trainer import Trainer

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    train_loader=train_loader,
    valid_loader=val_loader,
    device=device,
    experiment_config=experiment_config,
    hyperparams=hyperparams,
)

# Load checkpoint if specified in YAML
if experiment_config.resume_from is not None:
    resume_from = PROJECT_ROOT / experiment_config.resume_from
    trainer.load_checkpoint(resume_from)
    for g in trainer.optimizer.param_groups:
        g["lr"] = hyperparams["learning_rate"]

    print("learning_rate set to ", hyperparams["learning_rate"])


Resumed from epoch 100. Best validation loss: 0.1494
learning_rate set to  0.0001


In [9]:
trainer.fit()

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from C:\Users\joshua\_netrc.
wandb: Currently logged in as: joshuardh (joshuardh-mindanao-state-university) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Training: 100%|██████████| 180/180 [02:21<00:00,  1.27it/s]
c:\Users\joshua\vsjycode\csc173\transformers\TRACERS\.venv\Lib\site-packages\torch\nn\modules\transformer.py:533: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:181.)
  output = torch._nested_tensor_from_mask(
Validation: 100%|██████████| 91/91 [00:24<00:00,  3.64it/s]



Epoch 101/102
Train | Total: 0.1130 | NP: 0.0087 | NS: 0.0520
Val   | Total: 0.1561 | NP: 0.0138 | NS: 0.0598

TRAIN METRICS
Onset  | P: 0.8397 | R: 0.8472 | F1: 0.8434
Action | P: 0.9119 | R: 0.8930 | F1: 0.9023
Active Exact Accuracy: 0.4361
GT Active Rate: 4.97% | Predicted Active Rate: 4.87%

VAL METRICS
Onset  | P: 0.7599 | R: 0.7260 | F1: 0.7425
Action | P: 0.8436 | R: 0.8655 | F1: 0.8544
Active Exact Accuracy: 0.3853
GT Active Rate: 4.37% | Predicted Active Rate: 4.48%


Epochs:  50%|█████     | 1/2 [02:46<02:46, 166.87s/it]

Saved checkpoint: checkpoints\latest.pt


Epochs: 100%|██████████| 2/2 [05:15<00:00, 157.82s/it]
wandb: ERROR The nbformat package was not found. It is required to save notebook history.



Epoch 102/102
Train | Total: 0.1123 | NP: 0.0087 | NS: 0.0517
Val   | Total: 0.1559 | NP: 0.0138 | NS: 0.0594

TRAIN METRICS
Onset  | P: 0.8429 | R: 0.8475 | F1: 0.8452
Action | P: 0.9139 | R: 0.8952 | F1: 0.9045
Active Exact Accuracy: 0.4372
GT Active Rate: 4.98% | Predicted Active Rate: 4.88%

VAL METRICS
Onset  | P: 0.7592 | R: 0.7386 | F1: 0.7488
Action | P: 0.8505 | R: 0.8609 | F1: 0.8557
Active Exact Accuracy: 0.3891
GT Active Rate: 4.37% | Predicted Active Rate: 4.42%
Saved checkpoint: checkpoints\latest.pt


epoch,▁█
time/epoch_seconds,█▁
time/train_seconds,█▁
time/val_seconds,█▁
train/action_f1,▁█
train/action_precision,▁█
train/action_recall,▁█
train/active_exact_accuracy,▁█
train/gt_active_rate,▁█
train/np_loss,█▁
+18,...
